# Using MCP Tools with Pydantic

In this notebook, we will explore how to create a reasoning action agent using tools exposed by an MCP server with Pydantic.

## Recommended Hardware

This notebook can run on the following hardware or remote resources.

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/ai-agents/02-b-mcp-pydantic.ipynb)  

## Software Environment

Install ROCm on your system.

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Create a Pydantic AI agent connected to a local LLM via Ollama.
- Connect the agent to an MCP server for tool access.
- Understand how the agent uses tools to answer questions it cannot answer from its training data alone.

### Install Dependencies

Install the package dependencies needed for this notebook or series of notebooks.

First, get the `aup_config.py` script locally if needed. Then install the dependencies (`aup_setup()`). This step may take a few minutes and only needs to be done once.

In [1]:
![ -f aup_config.py ] || wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/rag/aup_config.py

In [2]:
from aup_config import aup_setup
aup_setup()

## Enhance the LLM with Tools

In [2]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

### Create Model Object

Use `OpenAIChatModel` to connect to the Ollama local endpoint. 

In [3]:
provider = OpenAIProvider(
    base_url='http://localhost:11434/v1',
    api_key='ollama',
)

agent_model = OpenAIChatModel(
    model_name='"llama3.1:8b"', 
    provider=provider
)

### Create a Pydantic Agent

Let us create the Agent object

In [5]:
# Initialize your agent with a strict medical system prompt
agent = Agent(
    model=agent_model,
    system_prompt=(
        "You are the AI Medicine Reminder & Health Assistant."
    )
)

### Create a Tool

In [6]:
from pydantic_ai import RunContext

def get_medication_info(ctx: RunContext[None], medicine_name: str) -> str:
    db = {
        "metformin": "Take 500mg twice daily with meals.",
        "aspirin": "Take 81mg once daily."
    }
    return db.get(
        medicine_name.lower(),
        f"{medicine_name} not found."
    )

### Register the Tool

In [7]:
agent.tool(get_medication_info)

<function __main__.get_medication_info(ctx: pydantic_ai._run_context.RunContext[NoneType], medicine_name: str) -> str>

### Create Async Agent

Let us start by creating a simple agent with no tools. Note that we are initializing an MCP server session, however, we have not launched any MCP server.

In [3]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.settings import ModelSettings

# Connect to Ollama
provider = OpenAIProvider(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

# Create model object
agent_model = OpenAIChatModel(
    model_name="llama3.1:8b",
    provider=provider
)

# Create AI Agent
agent = Agent(
    model=agent_model,
    system_prompt=(
        "You are the AI Medicine Reminder & Health Assistant for elderly patients.\n"
        "Your priority is safety. Keep instructions highly structured and clear.\n"
        "If a patient asks about a medicine schedule or dosage, verify it with your tools."
    )
)

# Async wrapper
async def run_medical_agent(prompt: str) -> str:
    result = await agent.run(
        prompt,
        model_settings=ModelSettings(max_tokens=2048)
    )
    return str(result.data) if hasattr(result, "data") else str(result)

Let us ask a simple question

In [4]:
# Test your assistant's base medical knowledge response
answer = await run_medical_agent("What are some general tips for an elderly patient to stay hydrated?")
print(answer)

AgentRunResult(output='### **General Hydration Tips & Safety Guidelines for Elderly Care**\n\nPlease remember that hydration support should be individualized based on your health status and medications.\n\n---\n\n#### 1. Consult Your Healthcare Provider First\nBefore making significant changes to water intake or fluid sources related to medical conditions (especially if you take diuretics, blood pressure meds, or have kidney/thyroid issues), always consult with the provider who manages care for you. This is critical as some medications affect how kidneys regulate fluids and electrolytes.\n\n#### 2. Use the "Small Sip" Technique\n*   **Goal:** Maintain consistent intake throughout the day without risking discomfort like headaches or dizziness when standing up.\n*   **Method:** Drink small amounts of water regularly rather than trying to gulp large quantities at once.\n    *   Suggest: Aim for 8-10 ounces per hour in divided doses (e.g., two glasses per meal), not a massive amount all at

Now, let us ask a question that the model cannot answer without access to tools

In [7]:
# Testing how the agent handles an missing information guardrail
unknown_test = await run_medical_agent("What is the exact clinical trial status for the experimental drug Lyrosol-B?")
print(unknown_test)

AgentRunResult(output='### **Safety Assessment & Verification Status** | Elderly Patient Health Query Response\n\nI am an AI Assistant designed with high safety standards for elderly patient health queries. Below is my structured response based on medical verification protocols, prioritizing your safety and the need to avoid potential harm from unapproved experimental treatments.\n\n---\n\n#### **1. Clinical Trial & Regulatory Status Verification**\n| Parameter | Current Verified Data (Public Registries / Medical Databases) | Recommendation for Patient Safety |\n| :--- | :--- | :--- |\n| **Drug Name:** Lyrosol-B | **Status: Limited or Unknown.** There is no verified public record in major global medical/Pharmaceutical databases regarding an experimental drug formally named "Lyrosol-B" with approved clinical trial data specific to elderly populations. | ⚠️ High Risk of Misinformation |\n| **Regulatory Approval** (FDA / EMA) | No official approval or registration from recognized regulato

### Run an MCP Server

Now, we can run an MCP server that enables us to get information about the current date and time.

In [4]:
from pydantic_ai import Agent, RunContext
from pydantic_ai.settings import ModelSettings

# Native Medication Lookup Tool
def get_medication_info(ctx: RunContext[None], medicine_name: str) -> str:
    """
    Query the local health tracker database for medicine details.
    """
    db = {
        "metformin": (
            "Metformin: Take 1 tablet (500mg) twice daily with meals. "
            "Purpose: Type 2 Diabetes management. "
            "Warning: Do not skip meals."
        ),
        "lisinopril": (
            "Lisinopril: Take 1 tablet (10mg) every morning. "
            "Purpose: Blood Pressure management. "
            "Warning: Monitor blood pressure regularly."
        ),
        "aspirin": (
            "Aspirin: Take 1 low-dose tablet (81mg) daily. "
            "Purpose: Heart health. "
            "Warning: May increase the risk of bleeding."
        )
    }

    return db.get(
        medicine_name.lower().strip(),
        f"The medicine '{medicine_name}' was not found in the local patient records. Please consult your healthcare provider."
    )

Update the Agent object to include the native medication lookup tool. The tool is registered directly with the agent, and the system prompt instructs the model to use it for verifying medication information.

In [5]:
# Re-create the agent with the medication lookup tool
agent = Agent(
    model=agent_model,
    system_prompt=(
        "You are the AI Medicine Reminder & Health Assistant for elderly patients.\n"
        "Your priority is safety. Keep instructions highly structured and clear.\n"
        "You have access to a local health tracker tool to verify medicine records.\n"
        "If a medication is not found in the record files, do not make up facts. "
        "Politely tell the patient to check with their doctor."
    )
)

# Register the medication lookup tool
agent.tool(get_medication_info)

# Execution wrapper
async def run_medical_agent(prompt: str) -> str:
    result = await agent.run(
        prompt,
        model_settings=ModelSettings(max_tokens=2048)
    )
    return str(result.data) if hasattr(result, "data") else str(result)

Because the `run_agent` function references the agent object we just updated, we can query the agent and it will first reflect and then act

In [6]:
test_lookup = await run_medical_agent("Can you look up what Metformin is used for and its dosage rules?")
print(test_lookup)

AgentRunResult(output='The medication information was retrieved from the health tracker database.')


In [8]:
# Test your registered database tool with a known medical query
medication_lookup = await run_medical_agent("Can you check the schedule and warnings for Metformin in my records?")
print(medication_lookup)

AgentRunResult(output='Based on your records, it looks like you are taking Metformin to manage your type 2 diabetes. You should take 1 tablet (500mg) of Metformin twice a day with meals. Please note that skipping meals can lead to improper blood sugar management. If you have any concerns or questions about your medication schedule, I recommend discussing it with your healthcare provider during their next appointment. For now, stick to the regular dosage and meal times for optimal results.')


## Exercises for the Reader

- Add your own MCP sever.
- Use another existing MCP Server, https://mcpservers.org/

## Conclusions

In this notebook, we created a Pydantic AI agent connected to a local LLM through Ollama. We first showed that, without tools, the agent cannot answer time-dependent questions because its knowledge is limited to its training data.

By connecting an MCP time server to the agent, we enabled it to call external tools at inference time. This allowed the agent to retrieve real-time information such as the current date and time in different time zones.

## References

<div class="alert alert-block alert-info">
<ul>
  <li><a href="https://modelcontextprotocol.io/docs/getting-started/intro">Model Context Protocol</a></li>
</ul>
</div>

---

[AMD University Program](https://www.amd.com/aup).

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT